# 🌿 Detección de Enfermedades en Hojas — YOL011

> **Google Colab** · GPU T4 Gratuita  
> Dataset: Plant Disease (6 clases) — Roboflow Public Domain

## 📋 Clases
| ID | Nombre técnico | Español |
|:--:|---|---|
| 0 | `bercak_daun` | Mancha Foliar |
| 1 | `defisiensi_kalsium` | Deficiencia de Calcio |
| 2 | `hangus_daun` | Quemadura de Hoja |
| 3 | `hawar_daun` | Tizón Foliar |
| 4 | `mosaik_vena_kuning` | Mosaico Vena Amarilla |
| 5 | `virus_kuning_keriting` | Virus Rizado Amarillo |

---
### ⚡ Antes de empezar
1. Ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**
2. Ejecuta las celdas **en orden**

## 1️⃣ Verificar GPU

In [ ]:
import torch

print('=== Información del entorno ===')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA disponible : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  GPU no disponible — ve a Entorno de ejecución → Cambiar tipo → GPU')

!nvidia-smi

## 2️⃣ Instalar Ultralytics (YOLO11)

In [ ]:
!pip install ultralytics -q

from ultralytics import YOLO
import ultralytics
print(f'✅ Ultralytics {ultralytics.__version__} instalado correctamente')

## 3️⃣ Montar Google Drive y subir el dataset

> ### 📦 ¿Cómo preparar el dataset?
> En tu computadora local, comprime la carpeta del proyecto:
> ```
> Comprime las carpetas: images/ y dataset/
> Archivo resultante: dataset_hojas.zip
> ```
> Luego sube `dataset_hojas.zip` a tu **Google Drive** (carpeta raíz o una subcarpeta).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado en /content/drive')

In [ ]:
import os

# ── AJUSTA ESTA RUTA al ZIP en tu Drive ──────────────────────────────
ZIP_PATH = '/content/drive/MyDrive/dataset_hojas.zip'
# ─────────────────────────────────────────────────────────────────────

WORK_DIR = '/content/leaf_disease'
os.makedirs(WORK_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f'No se encontró {ZIP_PATH}\n'
        f'Sube el ZIP a Google Drive y actualiza ZIP_PATH en esta celda.'
    )

print(f'📦 Descomprimiendo dataset...')
!unzip -q "{ZIP_PATH}" -d "{WORK_DIR}"
print(f'✅ Dataset extraído en {WORK_DIR}')

# Verificar estructura
!find "{WORK_DIR}" -maxdepth 3 -type d

## 4️⃣ Configurar data.yaml para Colab

In [ ]:
import yaml
from pathlib import Path

WORK_DIR = Path('/content/leaf_disease')

# Detectar estructura automáticamente
train_path = WORK_DIR / 'images' / 'train' / 'images'
val_path   = WORK_DIR / 'images' / 'valid' / 'images'
test_path  = WORK_DIR / 'images' / 'test'  / 'images'

print('Verificando rutas del dataset:')
for p in [train_path, val_path, test_path]:
    imgs = list(p.glob('*.jpg')) + list(p.glob('*.png'))
    status = '✅' if p.exists() else '❌'
    print(f'  {status} {p}  ({len(imgs)} imágenes)')

# Escribir data.yaml con rutas absolutas de Colab
data_yaml = {
    'path' : str(WORK_DIR),
    'train': 'images/train/images',
    'val'  : 'images/valid/images',
    'test' : 'images/test/images',
    'nc'   : 6,
    'names': [
        'bercak_daun',
        'defisiensi_kalsium',
        'hangus_daun',
        'hawar_daun',
        'mosaik_vena_kuning',
        'virus_kuning_keriting',
    ],
}

yaml_out = WORK_DIR / 'data.yaml'
with open(yaml_out, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print(f'\n✅ data.yaml generado: {yaml_out}')
print('--- Contenido ---')
!cat "{yaml_out}"

## 5️⃣ Entrenamiento YOLO11

| Parámetro | Valor | Descripción |
|---|---|---|
| `model` | `yolo11n.pt` | Nano (más rápido) |
| `epochs` | 50 | Iteraciones |
| `batch` | 32 | Imágenes por lote (ajustar si hay OOM) |
| `imgsz` | 640 | Tamaño de imagen |
| `device` | 0 | GPU T4 |

In [ ]:
from ultralytics import YOLO
from pathlib import Path

WORK_DIR   = Path('/content/leaf_disease')
DATA_YAML  = WORK_DIR / 'data.yaml'

# ── Hiperparámetros ────────────────────────────────────────────
MODEL_NAME = 'yolo11n.pt'   # nano: rápido | yolo11s.pt: más preciso
EPOCHS     = 50
BATCH      = 32             # reducir a 16 si aparece error de memoria
IMGSZ      = 640
DEVICE     = 0              # GPU T4
EXP_NAME   = 'leaf_disease_yolo11n'
# ───────────────────────────────────────────────────────────────

model = YOLO(MODEL_NAME)

results = model.train(
    data    = str(DATA_YAML),
    epochs  = EPOCHS,
    batch   = BATCH,
    imgsz   = IMGSZ,
    device  = DEVICE,
    project = str(WORK_DIR / 'runs' / 'train'),
    name    = EXP_NAME,
    patience = 20,

    # Augmentación
    hsv_h   = 0.015,
    hsv_s   = 0.7,
    hsv_v   = 0.4,
    flipud  = 0.2,
    fliplr  = 0.5,
    mosaic  = 1.0,
    mixup   = 0.15,

    plots   = True,
    verbose = True,
)

BEST_PT = Path(results.save_dir) / 'weights' / 'best.pt'
print(f'\n✅ Entrenamiento completo!')
print(f'📍 Mejor modelo: {BEST_PT}')
print(f'📊 mAP50 final : {results.results_dict.get("metrics/mAP50(B)", 0):.4f}')

## 6️⃣ Evaluación en conjunto de prueba

In [ ]:
from ultralytics import YOLO
from pathlib import Path

WORK_DIR  = Path('/content/leaf_disease')
DATA_YAML = WORK_DIR / 'data.yaml'

# Cargar mejor modelo
best_pt = sorted((WORK_DIR / 'runs' / 'train').rglob('best.pt'))[-1]
print(f'Cargando modelo: {best_pt}')
model = YOLO(str(best_pt))

metrics = model.val(
    data    = str(DATA_YAML),
    split   = 'test',
    conf    = 0.001,
    iou     = 0.6,
    device  = 0,
    plots   = True,
)

print('\n' + '='*50)
print('  📈 RESULTADOS EN CONJUNTO DE PRUEBA')
print('='*50)
print(f'  mAP50      : {metrics.box.map50:.4f}')
print(f'  mAP50-95   : {metrics.box.map:.4f}')
print(f'  Precision  : {metrics.box.mp:.4f}')
print(f'  Recall     : {metrics.box.mr:.4f}')
print('='*50)

CLASS_ES = [
    'Mancha Foliar', 'Deficiencia Calcio', 'Quemadura Hoja',
    'Tizón Foliar',  'Mosaico Vena Amarilla', 'Virus Rizado'
]
print(f'\n  {"Clase":<25} {"AP50":>8} {"Prec":>8} {"Recall":>8}')
print('  ' + '-'*55)
for i, (ap, p, r) in enumerate(zip(metrics.box.ap50, metrics.box.p, metrics.box.r)):
    print(f'  {CLASS_ES[i]:<25} {ap:>8.4f} {p:>8.4f} {r:>8.4f}')

## 7️⃣ Visualizar resultados de entrenamiento

In [ ]:
from IPython.display import Image, display
from pathlib import Path

WORK_DIR = Path('/content/leaf_disease')
run_dir  = sorted((WORK_DIR / 'runs' / 'train').glob('*'))[-1]

plots = [
    ('Curvas de Entrenamiento',  run_dir / 'results.png'),
    ('Matriz de Confusión',      run_dir / 'confusion_matrix.png'),
    ('Curva PR',                 run_dir / 'PR_curve.png'),
    ('Curva F1',                 run_dir / 'F1_curve.png'),
]

for title, path in plots:
    if path.exists():
        print(f'\n### {title}')
        display(Image(filename=str(path), width=800))
    else:
        print(f'No encontrado: {path}')

## 8️⃣ Prueba de inferencia en imágenes de test

In [ ]:
import random
from pathlib import Path
from IPython.display import Image as IPImage, display
from ultralytics import YOLO
import cv2

WORK_DIR = Path('/content/leaf_disease')
best_pt  = sorted((WORK_DIR / 'runs' / 'train').rglob('best.pt'))[-1]
model    = YOLO(str(best_pt))

test_images = list((WORK_DIR / 'images' / 'test' / 'images').glob('*.jpg'))
sample      = random.sample(test_images, min(6, len(test_images)))

CLASS_ES = [
    'Mancha Foliar', 'Deficiencia Calcio', 'Quemadura Hoja',
    'Tizón Foliar',  'Mosaico Vena Amarilla', 'Virus Rizado'
]

for img_path in sample:
    results = model.predict(str(img_path), conf=0.40, iou=0.45, verbose=False)
    annotated = results[0].plot()

    out = WORK_DIR / f'pred_{img_path.name}'
    cv2.imwrite(str(out), annotated)

    dets = []
    if results[0].boxes is not None:
        for box in results[0].boxes:
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            dets.append(f'{CLASS_ES[cls_id]} ({conf:.0%})')

    print(f'\n📸 {img_path.name}')
    print(f'   Detecciones: {", ".join(dets) if dets else "✅ Sin enfermedades"}')
    display(IPImage(filename=str(out), width=500))

## 9️⃣ Guardar modelo en Google Drive y descargar

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

WORK_DIR = Path('/content/leaf_disease')

# Encontrar mejor modelo
best_pt = sorted((WORK_DIR / 'runs' / 'train').rglob('best.pt'))[-1]

# ── Copiar a Google Drive ──────────────────────────────────────
drive_dest = Path('/content/drive/MyDrive/leaf_disease_best.pt')
shutil.copy(best_pt, drive_dest)
print(f'✅ Modelo guardado en Google Drive: {drive_dest}')

# ── Descargar directamente ─────────────────────────────────────
print('\n📥 Iniciando descarga directa...')
files.download(str(best_pt))
print('\n📌 Instrucciones finales:')
print('   1. Guarda best.pt en: YOLO/models/best.pt')
print('   2. Lanza la app con:  python app/app.py')

## 🏁 ¡Listo!

Una vez descargado `best.pt`:

```bash
# 1. Copia el modelo descargado a:
#    YOLO/models/best.pt

# 2. Lanza la aplicación web local
python app/app.py

# 3. Abre en el navegador
#    http://127.0.0.1:7860
```

---
> **Proyecto:** Detección de Enfermedades en Hojas · YOLO11 · SEXTO SEMESTRE